# 1. Setup e Carga de Dados

In [ ]:
import os
import json
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive

drive.mount('/content/drive')

caminho_pasta = '/content/drive/MyDrive/Projetos/RecomendacaoSpotify'
arquivos = [
    os.path.join(caminho_pasta, 'Streaming_History_Audio_2019-2024_0.json'),
    os.path.join(caminho_pasta, 'Streaming_History_Audio_2024-2026_1.json'),
    os.path.join(caminho_pasta, 'Streaming_History_Audio_2026.json')
]

dataframes = []
ultimo_ts = None

#Carrega a partir do primeiro arquivo, e vai incrementando com os demais,
#a partir da data de corte do arquivo anterior

for arquivo in arquivos:
    if os.path.exists(arquivo):
        with open(arquivo, 'r', encoding='utf-8') as f:
            df_temp = pd.DataFrame(json.load(f))
            if not df_temp.empty:
                if ultimo_ts is not None:
                    df_temp = df_temp[df_temp['ts'] > ultimo_ts]
                if not df_temp.empty:
                    ultimo_ts = df_temp['ts'].max()
                    dataframes.append(df_temp)

df_spotify = pd.concat(dataframes, ignore_index=True).drop_duplicates(ignore_index=True)
print(f"Total de registros carregados: {len(df_spotify)}")

# Primeira versão do arquivo csv do mapeamento de artistas:
# original,unificado
# Engenheiros Do Hawaii,Engenheiros e Humberto Gessinger
# Pouca Vogal,Engenheiros e Humberto Gessinger
# Humberto Gessinger,Engenheiros e Humberto Gessinger
# The Police,Sting e The Police
# Sting,Sting e The Police
# Liniker e os Caramelows,Liniker

caminho_mapeamento_artistas = os.path.join(caminho_pasta, 'mapeamento_artistas.csv')

if os.path.exists(caminho_mapeamento_artistas):
    df_mapeamento = pd.read_csv(caminho_mapeamento_artistas)
    with open(caminho_mapeamento_artistas) as f:
        mapeamento_artistas = dict(zip(df_mapeamento['original'], df_mapeamento['unificado']))
        print(f'Mapeamento carregado: {len(mapeamento_artistas)} entradas')
else:
    mapeamento_artistas = {}
    print('Arquivo de mapeamento não encontrado — lista vazia.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Total de registros carregados: 31515
Mapeamento carregado: 6 entradas


# 2. Engenharia de Atributos, Padronização de músicas e artistas e Separação de Músicas

In [ ]:
# Datas e Tempo
df_spotify['data_finalizacao'] = pd.to_datetime(df_spotify['ts'])
df_spotify['ano'] = df_spotify['data_finalizacao'].dt.year
df_spotify['mes'] = df_spotify['data_finalizacao'].dt.month
df_spotify['hora'] = df_spotify['data_finalizacao'].dt.hour
df_spotify['semana_ano'] = df_spotify['data_finalizacao'].dt.isocalendar().week.astype(int)
df_spotify['dia_semana']    = df_spotify['data_finalizacao'].dt.day_name()
df_spotify['trimestre']     = df_spotify['data_finalizacao'].dt.quarter
df_spotify['periodo_dia'] = pd.cut(
    df_spotify['hora'],
    bins=[-1, 5, 10, 18, 23],
    labels=['madrugada', 'manha', 'tarde', 'noite'])

#0h - 5h - Madrugada / 06h - 10h - Manhã / 11h - 18h - Tarde / 19h - 23h - Noite

df_spotify['minutos_tocados'] = df_spotify['ms_played'] / 60000

def simplificar_nome_musica(nome):
    if pd.isna(nome): return nome
    termos_versao = r'remaster|live|ao vivo|version|edit|anniversary|deluxe|digital|mix|stereo|mono|feat|acústico|acoustic'
    nome = re.sub(r'\s*[\[\(](?=[^\)\]]*(?:' + termos_versao + r'))[^\)\]]*[\]\)]\s*$', '', nome, flags=re.IGNORECASE)
    #nome = re.sub(r'\s*[\-\–\—]\s*(?=.*(?:' + termos_versao + r'|\d{4})).*$', '', nome, flags=re.IGNORECASE)
    nome = re.sub(r'\s+[\-\–\—]\s+(?=.*(?:' + termos_versao + r'|\d{4})).*$', '', nome, flags=re.IGNORECASE)
    return re.sub(r'\s+', ' ', nome).strip()

df_spotify['musica_unificada'] = df_spotify['master_metadata_track_name'].apply(simplificar_nome_musica)

df_spotify['artista_unificado'] = df_spotify['master_metadata_album_artist_name'].replace(mapeamento_artistas)

print(f"Músicas únicas antes de simplificação: {df_spotify['master_metadata_track_name'].nunique()}")
print(f"Músicas únicas após simplificação: {df_spotify['musica_unificada'].nunique()}")
print(f"Artistas únicos antes de simplificação: {df_spotify['master_metadata_album_artist_name'].nunique()}")
print(f"Artistas únicos após unificação: {df_spotify['artista_unificado'].nunique()}")

#Removendo dados pessoais e dados inúteis
cols_remover_spotify = [
    'platform', 'conn_country', 'ip_addr', 'offline_timestamp', 'incognito_mode'
]

df_spotify = df_spotify.drop(columns=[c for c in cols_remover_spotify if c in df_spotify.columns])

# Filtrando apenas músicas e removendo colunas inúteis para músicas
df_musicas = df_spotify[df_spotify['master_metadata_track_name'].notnull()].copy()
cols_remover = [
    'episode_name', 'episode_show_name', 'spotify_episode_uri',
    'audiobook_title', 'audiobook_uri', 'audiobook_chapter_uri',
    'audiobook_chapter_title'
]
df_musicas = df_musicas.drop(columns=[c for c in cols_remover if c in df_musicas.columns])

print(f"Total após engenharia de atributos: {len(df_musicas)} registros de música.")

Músicas únicas antes de simplificação: 8340
Músicas únicas após simplificação: 7650
Artistas únicos antes de simplificação: 1406
Artistas únicos após unificação: 1402
Total após engenharia de atributos: 28933 registros de música.


In [ ]:
display(df_spotify.info())
display(df_musicas.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31515 entries, 0 to 31514
Data columns (total 29 columns):
 #   Column                             Non-Null Count  Dtype              
---  ------                             --------------  -----              
 0   ts                                 31515 non-null  object             
 1   ms_played                          31515 non-null  int64              
 2   master_metadata_track_name         28933 non-null  object             
 3   master_metadata_album_artist_name  28933 non-null  object             
 4   master_metadata_album_album_name   28933 non-null  object             
 5   spotify_track_uri                  28933 non-null  object             
 6   episode_name                       2581 non-null   object             
 7   episode_show_name                  2581 non-null   object             
 8   spotify_episode_uri                2581 non-null   object             
 9   audiobook_title                    0 non-null     

None

<class 'pandas.core.frame.DataFrame'>
Index: 28933 entries, 0 to 31513
Data columns (total 22 columns):
 #   Column                             Non-Null Count  Dtype              
---  ------                             --------------  -----              
 0   ts                                 28933 non-null  object             
 1   ms_played                          28933 non-null  int64              
 2   master_metadata_track_name         28933 non-null  object             
 3   master_metadata_album_artist_name  28933 non-null  object             
 4   master_metadata_album_album_name   28933 non-null  object             
 5   spotify_track_uri                  28933 non-null  object             
 6   reason_start                       28933 non-null  object             
 7   reason_end                         28933 non-null  object             
 8   shuffle                            28933 non-null  bool               
 9   skipped                            28933 non-null  bool

None

# 3. Rating de Lealdade e Filtros de Qualidade

In [ ]:
# Agregação por Artista para Rating
data_ref = df_musicas['data_finalizacao'].max()
df_artistas = df_musicas.groupby('artista_unificado').agg(
    total_plays_artista=('ts', 'count'),
    primeiro_play_artista=('data_finalizacao', 'min'),
    ultimo_play_artista=('data_finalizacao', 'max')
).reset_index()

df_artistas['longevidade_dias_artista'] = (df_artistas['ultimo_play_artista'] - df_artistas['primeiro_play_artista']).dt.days
df_artistas['dias_sem_ouvir_artista'] = (data_ref - df_artistas['ultimo_play_artista']).dt.days
df_artistas['rating_lealdade_artista'] = np.log10(
    ((df_artistas['total_plays_artista']**2) * np.log1p(df_artistas['longevidade_dias_artista']) / np.log1p(df_artistas['dias_sem_ouvir_artista'] + 1)) + 1
)
# Aplicando Filtros. Decisão para evitar chamadas extras à api do spotify
artistas_relevantes = df_artistas[df_artistas['rating_lealdade_artista'] >= 1.0]['artista_unificado']
df_musicas_limpo = df_musicas[
    (df_musicas['artista_unificado'].isin(artistas_relevantes)) &
    (df_musicas['ms_played'] >= 1000)
].copy()

# Ranking Final (Mínimo 2 minutos acumulados).
df_ranking_final = df_musicas_limpo.groupby(['artista_unificado', 'musica_unificada']).agg(
    total_minutos=('minutos_tocados', 'sum'),
    total_plays=('ts', 'count'),
).reset_index().query('total_minutos >= 2.0').sort_values('total_minutos', ascending=False)

print(f"Artistas Relevantes (Rating >= 1.0): {len(artistas_relevantes)}")
print(f"Músicas no Ranking Final (>= 2min): {len(df_ranking_final)}")
display(df_ranking_final.head(10))

Artistas Relevantes (Rating >= 1.0): 478
Músicas no Ranking Final (>= 2min): 4212


,artista_unificado,musica_unificada,total_minutos,total_plays
4375,Pink Floyd,Echoes,267.779633,27
2770,Johnny Hooker,Amor Marginal,260.614650,64
4635,Plebe Rude,O Pêndulo da História,247.048983,33
377,Arnaldo Antunes,Meu Coração,229.937417,43
579,Bad Religion,Generator,227.890383,86
618,Bad Religion,Los Angeles Is Burning,216.031883,68
603,Bad Religion,Infected,201.040533,69
69,ANGRA,Carolina IV,195.959667,30
631,Bad Religion,My Sanity,191.290733,77
2398,Green Day,Jesus of Suburbia,181.900833,27


## 5. Enriquecimento do Dataset de Músicas com dados dos artistas
Cruzando o ranking final com o histórico completo e metadados dos artistas.

In [ ]:
# Filtrar df_musicas pelas músicas relevantes do ranking
chaves_validas = df_ranking_final[['artista_unificado', 'musica_unificada']].drop_duplicates()
df_musicas = df_musicas.merge(chaves_validas, on=['artista_unificado', 'musica_unificada'], how='inner')

# Trazer features de artista
cols_artistas = ['artista_unificado', 'rating_lealdade_artista', 'longevidade_dias_artista', 'total_plays_artista']
df_musicas = df_musicas.merge(df_artistas[cols_artistas], on='artista_unificado', how='left')

print(f"Registros após filtro de relevância: {len(df_musicas)}")
print(f"Músicas únicas mantidas: {df_musicas['musica_unificada'].nunique()}")
print(f"Artistas únicos mantidos: {df_musicas['artista_unificado'].nunique()}")

Registros após filtro de relevância: 24040
Músicas únicas mantidas: 4047
Artistas únicos mantidos: 438


# 5. Enriquecimento do Dataset de Músicas com dados de Sessão

In [ ]:
# ── Período do dia ─────────────────────────────────────────────────────────────
# [0h,  5h] → madrugada | [6h, 10h] → manha | [11h, 18h] → tarde | [19h, 23h] → noite
for df in [df_spotify, df_musicas]:
    df['periodo_dia'] = pd.cut(
        df['hora'],
        bins=[-1, 5, 10, 18, 23],
        labels=['madrugada', 'manha', 'tarde', 'noite']
    )

# ── Sessões ────────────────────────────────────────────────────────────────────
# Dia ajustado: começa às 6h (madrugada pertence ao dia anterior)
# Gap de 45 min ou mudança de dia ajustado define nova sessão
# sessao_id: contador por dia ajustado (1, 2, 3...)

df_musicas = df_musicas.sort_values('data_finalizacao').reset_index(drop=True)

df_musicas['dia_ajustado'] = (
    df_musicas['data_finalizacao'] - pd.Timedelta(hours=6)
).dt.date.astype(str)

df_musicas['gap_anterior'] = (
    df_musicas['data_finalizacao'] - df_musicas['data_finalizacao'].shift(1)
).dt.total_seconds() / 60

nova_sessao = (
    (df_musicas['gap_anterior'] > 45) |
    (df_musicas['dia_ajustado'] != df_musicas['dia_ajustado'].shift(1))
)

df_musicas['sessao_id'] = (
    nova_sessao.groupby(df_musicas['dia_ajustado']).cumsum().astype(int)
)

# ── Features de posição ────────────────────────────────────────────────────────
# posicao_relativa_na_sessao: 0.0 = início, 1.0 = fim. Sessão de 1 música = 0.0
# posicao_sessao_no_dia:      0.0 = primeira sessão, 1.0 = última.
#                             Dia com sessão única usa horário como proxy.

for col in ['posicao_relativa_na_sessao', 'posicao_sessao_no_dia']:
    if col in df_musicas.columns:
        df_musicas = df_musicas.drop(columns=col)

def posicao_na_sessao(grupo):
    n = len(grupo)
    if n == 1:
        return pd.Series([0.0], index=grupo.index)
    return pd.Series([i / (n - 1) for i in range(n)], index=grupo.index)

df_musicas['posicao_relativa_na_sessao'] = (
    df_musicas.groupby(['dia_ajustado', 'sessao_id'], group_keys=False)['data_finalizacao']
    .apply(posicao_na_sessao)
)

def posicao_no_dia(grupo):
    n = len(grupo)
    if n == 1:
        hora_aj = (grupo.iloc[0] - 6) % 24
        return pd.Series([hora_aj / 23], index=grupo.index)
    return pd.Series([i / (n - 1) for i in range(n)], index=grupo.index)

sessoes_por_dia = (
    df_musicas.groupby(['dia_ajustado', 'sessao_id'])['hora']
    .first()
    .reset_index()
)

sessoes_por_dia['posicao_sessao_no_dia'] = (
    sessoes_por_dia.groupby('dia_ajustado', group_keys=False)['hora']
    .apply(posicao_no_dia)
)

df_musicas = df_musicas.merge(
    sessoes_por_dia[['dia_ajustado', 'sessao_id', 'posicao_sessao_no_dia']],
    on=['dia_ajustado', 'sessao_id'],
    how='left'
)

In [ ]:
df_musicas[['dia_ajustado', 'sessao_id', 'artista_unificado', 'musica_unificada', 'data_finalizacao', 'posicao_relativa_na_sessao', 'posicao_sessao_no_dia']].tail(10)

,dia_ajustado,sessao_id,artista_unificado,musica_unificada,data_finalizacao,posicao_relativa_na_sessao,posicao_sessao_no_dia
24030,2026-04-30,3,Garotos Podres,A Internacional,2026-05-01 03:42:18+00:00,0.00,1.0
24031,2026-04-30,3,The Strokes,Juicebox,2026-05-01 03:42:59+00:00,0.25,1.0
24032,2026-04-30,3,The Strokes,Juicebox,2026-05-01 03:43:10+00:00,0.50,1.0
24033,2026-04-30,3,The Strokes,Juicebox,2026-05-01 03:43:22+00:00,0.75,1.0
24034,2026-04-30,3,The Strokes,Juicebox,2026-05-01 03:43:35+00:00,1.00,1.0
24035,2026-05-01,1,Camel,Air Born,2026-05-01 13:00:07+00:00,0.00,0.0
24036,2026-05-01,1,Camel,Air Born,2026-05-01 13:00:07+00:00,0.50,0.0
24037,2026-05-01,1,Black Pantera,Estandarte,2026-05-01 13:08:18+00:00,1.00,0.0
24038,2026-05-01,2,Garotos Podres,Surfista de Pinico,2026-05-01 17:09:51+00:00,0.00,1.0
24039,2026-05-01,2,Garotos Podres,Anarkia Oi!,2026-05-01 17:11:41+00:00,1.00,1.0


In [ ]:
df_musicas.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24040 entries, 0 to 24039
Data columns (total 30 columns):
 #   Column                             Non-Null Count  Dtype              
---  ------                             --------------  -----              
 0   ts                                 24040 non-null  object             
 1   ms_played                          24040 non-null  int64              
 2   master_metadata_track_name         24040 non-null  object             
 3   master_metadata_album_artist_name  24040 non-null  object             
 4   master_metadata_album_album_name   24040 non-null  object             
 5   spotify_track_uri                  24040 non-null  object             
 6   reason_start                       24040 non-null  object             
 7   reason_end                         24040 non-null  object             
 8   shuffle                            24040 non-null  bool               
 9   skipped                            24040 non-null 

# 6 - Exportações


In [ ]:
# Salvando dataframes principais
df_spotify.to_parquet(f'{caminho_pasta}/df_spotify.parquet', index=False)
df_musicas.to_parquet(f'{caminho_pasta}/df_musicas.parquet', index=False)

print(f'df_spotify: {df_spotify.shape}')
print(f'df_musicas: {df_musicas.shape}')

df_spotify: (31515, 29)
df_musicas: (24040, 30)


In [ ]:
# Gerando o DF de URIs únicas diretamente do histórico enriquecido
# Como df_musicas_enriquecido já foi filtrado pelo ranking final, ele é a fonte ideal

df_export_uris = df_musicas[['artista_unificado', 'musica_unificada', 'spotify_track_uri']].drop_duplicates(subset='spotify_track_uri')
df_export_uris = df_export_uris.sort_values(by=['artista_unificado', 'musica_unificada']).reset_index(drop=True)

print(f"Arquivo de busca gerado com {len(df_export_uris)} URIs únicas (Baseado no histórico enriquecido).")

# Exportando para CSV
df_export_uris.to_csv(f'{caminho_pasta}/uris_unicas5.csv', index=False)
print(f"Arquivo salvo em: {caminho_pasta}/uris_unicas5.csv")

Arquivo de busca gerado com 5374 URIs únicas (Baseado no histórico enriquecido).
Arquivo salvo em: /content/drive/MyDrive/Projetos/RecomendacaoSpotify/uris_unicas5.csv
